
# Building an RNN Manually in PyTorch

This notebook implements a basic recurrent neural network **without using**:

- `nn.RNN`
- `nn.RNNCell`
- `nn.LSTM`
- `nn.GRU`

The goal is to understand exactly what happens inside an RNN at every time step.

We will manually create and use:

$$
W_{xh}
$$

the input-to-hidden weight matrix,

$$
W_{hh}
$$

the hidden-to-hidden recurrent weight matrix, and

$$
W_{hy}
$$

the hidden-to-output weight matrix.

By the end, you will understand:

1. Why an RNN needs a hidden state.
2. Why $W_{hh}$ has shape `hidden_size × hidden_size`.
3. How the same parameters are reused at every time step.
4. How words are represented as inputs.
5. How the RNN produces a sequence of hidden states and outputs.
6. How backpropagation trains all the manually created weight matrices.



## 1. Why do we need an RNN?

A normal feed-forward neural network treats every input independently.

For example, suppose we process these words:

```text
I like deep learning
```

A feed-forward network may process the word `learning` without directly remembering the words that appeared before it.

However, language is sequential. The meaning of a word often depends on earlier words.

An RNN solves this by maintaining a **hidden state**.

The hidden state acts like a numerical memory:

$$
h_t
$$

At time step $t$, the RNN uses:

- the current input $x_t$
- the previous hidden state $h_{t-1}$

to calculate a new hidden state:

$$
h_t = \tanh(x_tW_{xh} + h_{t-1}W_{hh} + b_h)
$$

The new hidden state therefore contains information from:

- the current word
- the previous hidden state
- indirectly, all earlier words



## 2. Our example dimensions

We will use:

- Vocabulary size: `5`
- Sequence length: `5`
- Hidden size: `3`
- Output size: `5`
- Batch size: `1`

Our vocabulary is:

```text
I
like
deep
learning
.
```

Each word is initially represented using a one-hot vector of length `5`.

For example:

$$
\text{"I"} = [1,0,0,0,0]
$$

$$
\text{"like"} = [0,1,0,0,0]
$$

At a single time step:

$$
x_t \in \mathbb{R}^{1 \times 5}
$$

The hidden state contains three values:

$$
h_t \in \mathbb{R}^{1 \times 3}
$$



## 3. Why does $W_{xh}$ have shape $5 \times 3$?

The current input has shape:

$$
x_t: 1 \times 5
$$

We want the input contribution to produce one value for each of the three hidden neurons.

Therefore:

$$
W_{xh}: 5 \times 3
$$

Then:

$$
(1 \times 5)(5 \times 3) = 1 \times 3
$$

So:

$$
x_tW_{xh}
$$

produces a vector with three values, one for each hidden neuron.



## 4. Why does $W_{hh}$ have shape $3 \times 3$?

The previous hidden state has shape:

$$
h_{t-1}: 1 \times 3
$$

We want to transform it into another vector with three hidden features.

Therefore:

$$
W_{hh}: 3 \times 3
$$

Then:

$$
(1 \times 3)(3 \times 3) = 1 \times 3
$$

The first `3` represents the number of features in the previous hidden state.

The second `3` represents the number of neurons in the current hidden state.

The previous hidden state does **not** become the matrix $W_{hh}$.

Instead:

- $h_{t-1}$ is changing data.
- $W_{hh}$ is a learned parameter.
- $h_{t-1}W_{hh}$ is the transformed previous memory.

The same $W_{hh}$ is reused at every time step.


In [10]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.13.0+cu130



## 5. Define the vocabulary

Each word receives an integer index.

These indices will later be converted into one-hot vectors.


In [11]:

vocabulary = {
    "I": 0,
    "like": 1,
    "deep": 2,
    "learning": 3,
    ".": 4,
}

index_to_word = {
    index: word for word, index in vocabulary.items()
}

vocabulary_size = len(vocabulary)

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", vocabulary)
index_to_word


Vocabulary size: 5
Vocabulary: {'I': 0, 'like': 1, 'deep': 2, 'learning': 3, '.': 4}


{0: 'I', 1: 'like', 2: 'deep', 3: 'learning', 4: '.'}


## 6. Create a sequence

Our complete sequence is:

```text
I like deep learning .
```

The integer representation is:

```text
0, 1, 2, 3, 4
```

The tensor initially has shape:

$$
(\text{batch size}, \text{sequence length})
$$

For one sequence containing five words:

$$
(1,5)
$$


In [12]:

sequence_indices = torch.tensor([
    [
        vocabulary["I"],
        vocabulary["like"],
        vocabulary["deep"],
        vocabulary["learning"],
        vocabulary["."],
    ]
])

print("Sequence indices:")
print(sequence_indices)

print("\nShape:", sequence_indices.shape)


Sequence indices:
tensor([[0, 1, 2, 3, 4]])

Shape: torch.Size([1, 5])



## 7. Convert the words into one-hot vectors

The RNN cannot directly multiply integer word indices by its weight matrix.

Each word must be represented as a feature vector.

For this educational example, we use one-hot encoding.

After conversion, the input shape becomes:

$$
(\text{batch size}, \text{sequence length}, \text{input size})
$$

In our case:

$$
(1,5,5)
$$

This means:

- one sequence
- five time steps
- five input features per word


In [13]:

x = F.one_hot(
    sequence_indices,
    num_classes=vocabulary_size
).float()

print("One-hot encoded input:")
print(x)

print("\nInput shape:", x.shape)


One-hot encoded input:
tensor([[[1., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0.],
         [0., 0., 1., 0., 0.],
         [0., 0., 0., 1., 0.],
         [0., 0., 0., 0., 1.]]])

Input shape: torch.Size([1, 5, 5])



## 8. Create the RNN parameters manually

We now create the trainable parameters ourselves.

### Input-to-hidden matrix

$$
W_{xh}: 5 \times 3
$$

It converts the current word vector into a hidden-sized vector.

### Hidden-to-hidden matrix

$$
W_{hh}: 3 \times 3
$$

It transforms the previous hidden state before that state contributes to the current hidden state.

### Hidden bias

$$
b_h: 3
$$

One bias value is added to each hidden neuron.

### Hidden-to-output matrix

$$
W_{hy}: 3 \times 5
$$

It converts the current hidden state into five output scores, one score for each vocabulary word.

These output values are called **logits**.


In [14]:

input_size = 5
hidden_size = 3
output_size = 5
batch_size = 1

W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.1)
W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
b_h = nn.Parameter(torch.zeros(hidden_size))

W_hy = nn.Parameter(torch.randn(hidden_size, output_size) * 0.1)
b_y = nn.Parameter(torch.zeros(output_size))

print("W_xh shape:", W_xh.shape)
print("W_hh shape:", W_hh.shape)
print("b_h shape :", b_h.shape)
print("W_hy shape:", W_hy.shape)
print("b_y shape :", b_y.shape)


W_xh shape: torch.Size([5, 3])
W_hh shape: torch.Size([3, 3])
b_h shape : torch.Size([3])
W_hy shape: torch.Size([3, 5])
b_y shape : torch.Size([5])



## 9. Initialize the hidden state

Before the first word is processed, there is no previous memory.

We therefore usually initialize:

$$
h_0 = 0
$$

Its shape must be:

$$
(\text{batch size}, \text{hidden size})
$$

For this example:

$$
(1,3)
$$


In [15]:

hidden = torch.zeros(batch_size, hidden_size)

print("Initial hidden state:")
print(hidden)

print("\nShape:", hidden.shape)


Initial hidden state:
tensor([[0., 0., 0.]])

Shape: torch.Size([1, 3])



## 10. Process only the first time step

At the first time step:

$$
x_1
$$

represents the word `I`.

We calculate two separate contributions.

### Current input contribution

$$
x_1W_{xh}
$$

### Previous hidden-state contribution

$$
h_0W_{hh}
$$

They both produce shape:

$$
1 \times 3
$$

so they can be added.

Then we apply `tanh`:

$$
h_1 = \tanh(x_1W_{xh} + h_0W_{hh} + b_h)
$$

`tanh` adds nonlinearity and keeps the hidden-state values between `-1` and `1`.


In [16]:

x_1 = x[:, 0, :]

input_part = x_1 @ W_xh
hidden_part = hidden @ W_hh

hidden_1 = torch.tanh(
    input_part + hidden_part + b_h
)

print("x_1 shape:", x_1.shape)
print("x_1 @ W_xh shape:", input_part.shape)
print("h_0 @ W_hh shape:", hidden_part.shape)
print("h_1 shape:", hidden_1.shape)

print("\nCurrent input x_1:")
print(x_1)

print("\nInput contribution:")
print(input_part)

print("\nPrevious hidden contribution:")
print(hidden_part)

print("\nNew hidden state h_1:")
print(hidden_1)


x_1 shape: torch.Size([1, 5])
x_1 @ W_xh shape: torch.Size([1, 3])
h_0 @ W_hh shape: torch.Size([1, 3])
h_1 shape: torch.Size([1, 3])

Current input x_1:
tensor([[1., 0., 0., 0., 0.]])

Input contribution:
tensor([[0.0337, 0.0129, 0.0234]], grad_fn=<MmBackward0>)

Previous hidden contribution:
tensor([[0., 0., 0.]], grad_fn=<MmBackward0>)

New hidden state h_1:
tensor([[0.0337, 0.0129, 0.0234]], grad_fn=<TanhBackward0>)



Because $h_0$ contains only zeros:

$$
h_0W_{hh}=0
$$

So at the first time step, the new hidden state mainly depends on the first input.

At later time steps, the recurrent contribution will usually be nonzero.



## 11. Produce an output from the hidden state

The hidden state is the RNN's internal representation.

To make a prediction, we convert it into output logits:

$$
y_1 = h_1W_{hy} + b_y
$$

Shapes:

$$
(1 \times 3)(3 \times 5) = 1 \times 5
$$

The resulting five values are unnormalized scores for the five possible vocabulary words.


In [17]:

output_1 = hidden_1 @ W_hy + b_y

probabilities_1 = torch.softmax(output_1, dim=-1)

print("Output logits shape:", output_1.shape)
print("Output logits:")
print(output_1)

print("\nProbabilities:")
print(probabilities_1)


Output logits shape: torch.Size([1, 5])
Output logits:
tensor([[-7.5551e-04, -1.9053e-03,  3.0494e-03, -1.1204e-03,  3.7614e-05]],
       grad_fn=<AddBackward0>)

Probabilities:
tensor([[0.1999, 0.1996, 0.2006, 0.1998, 0.2000]], grad_fn=<SoftmaxBackward0>)



## 12. Process the entire sequence manually

Now we repeat the same calculations for every time step.

The weight matrices do not change during the forward pass:

- the same $W_{xh}$ is used for every word
- the same $W_{hh}$ is used for every hidden transition
- the same $W_{hy}$ is used for every output

The hidden state does change:

$$
h_0 \rightarrow h_1 \rightarrow h_2 \rightarrow h_3 \rightarrow h_4 \rightarrow h_5
$$

This is called **parameter sharing across time**.

It allows an RNN to process sequences of different lengths without needing separate weights for each position.


In [18]:

hidden = torch.zeros(batch_size, hidden_size)

all_hidden_states = []
all_outputs = []

sequence_length = x.shape[1]

for time_step in range(sequence_length):
    x_t = x[:, time_step, :]

    input_part = x_t @ W_xh
    recurrent_part = hidden @ W_hh

    hidden = torch.tanh(
        input_part + recurrent_part + b_h
    )

    output_t = hidden @ W_hy + b_y

    all_hidden_states.append(hidden)
    all_outputs.append(output_t)

    current_word_index = sequence_indices[0, time_step].item()
    current_word = index_to_word[current_word_index]

    print(f"Time step {time_step + 1}")
    print("Current word:", current_word)
    print("x_t shape:", x_t.shape)
    print("Previous-memory contribution shape:", recurrent_part.shape)
    print("New hidden shape:", hidden.shape)
    print("Output shape:", output_t.shape)
    print("-" * 50)


Time step 1
Current word: I
x_t shape: torch.Size([1, 5])
Previous-memory contribution shape: torch.Size([1, 3])
New hidden shape: torch.Size([1, 3])
Output shape: torch.Size([1, 5])
--------------------------------------------------
Time step 2
Current word: like
x_t shape: torch.Size([1, 5])
Previous-memory contribution shape: torch.Size([1, 3])
New hidden shape: torch.Size([1, 3])
Output shape: torch.Size([1, 5])
--------------------------------------------------
Time step 3
Current word: deep
x_t shape: torch.Size([1, 5])
Previous-memory contribution shape: torch.Size([1, 3])
New hidden shape: torch.Size([1, 3])
Output shape: torch.Size([1, 5])
--------------------------------------------------
Time step 4
Current word: learning
x_t shape: torch.Size([1, 5])
Previous-memory contribution shape: torch.Size([1, 3])
New hidden shape: torch.Size([1, 3])
Output shape: torch.Size([1, 5])
--------------------------------------------------
Time step 5
Current word: .
x_t shape: torch.Size([


## 13. Stack all time-step results

During the loop, each hidden state has shape:

$$
(\text{batch size}, \text{hidden size})
$$

We stack them across the time dimension.

The complete hidden-state tensor becomes:

$$
(\text{batch size}, \text{sequence length}, \text{hidden size})
$$

For our example:

$$
(1,5,3)
$$

The output tensor becomes:

$$
(\text{batch size}, \text{sequence length}, \text{output size})
$$

For our example:

$$
(1,5,5)
$$


In [19]:

hidden_states_tensor = torch.stack(all_hidden_states, dim=1)
outputs_tensor = torch.stack(all_outputs, dim=1)

print("All hidden states shape:", hidden_states_tensor.shape)
print("All outputs shape:", outputs_tensor.shape)

print("\nFinal hidden state:")
print(hidden_states_tensor[:, -1, :])


All hidden states shape: torch.Size([1, 5, 3])
All outputs shape: torch.Size([1, 5, 5])

Final hidden state:
tensor([[ 0.1157, -0.1697, -0.0751]], grad_fn=<SelectBackward0>)



## 14. Put the logic inside a reusable PyTorch module

We will now create a proper model class.

Using `nn.Module` is useful because PyTorch can then:

- find all trainable parameters using `model.parameters()`
- move parameters to a GPU using `model.to(device)`
- calculate gradients automatically
- save and load the model
- work with PyTorch optimizers

We still do not use `nn.RNN`.


In [20]:

class ManualRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        # Current input -> current hidden state
        self.W_xh = nn.Parameter(
            torch.randn(input_size, hidden_size) * 0.1
        )

        # Previous hidden state -> current hidden state
        self.W_hh = nn.Parameter(
            torch.randn(hidden_size, hidden_size) * 0.1
        )

        self.b_h = nn.Parameter(
            torch.zeros(hidden_size)
        )

        # Current hidden state -> output logits
        self.W_hy = nn.Parameter(
            torch.randn(hidden_size, output_size) * 0.1
        )

        self.b_y = nn.Parameter(
            torch.zeros(output_size)
        )

    def forward(self, x, hidden=None, return_debug=False):
        if x.ndim != 3:
            raise ValueError(
                "Expected x to have shape "
                "(batch_size, sequence_length, input_size)"
            )

        batch_size, sequence_length, received_input_size = x.shape

        if received_input_size != self.input_size:
            raise ValueError(
                f"Expected input size {self.input_size}, "
                f"but received {received_input_size}"
            )

        if hidden is None:
            hidden = torch.zeros(
                batch_size,
                self.hidden_size,
                device=x.device,
                dtype=x.dtype
            )

        outputs = []
        hidden_states = []
        debug_information = []

        for time_step in range(sequence_length):
            x_t = x[:, time_step, :]

            input_part = x_t @ self.W_xh
            recurrent_part = hidden @ self.W_hh

            hidden = torch.tanh(
                input_part + recurrent_part + self.b_h
            )

            output_t = hidden @ self.W_hy + self.b_y

            hidden_states.append(hidden)
            outputs.append(output_t)

            if return_debug:
                debug_information.append({
                    "time_step": time_step,
                    "x_t": x_t.detach().clone(),
                    "input_part": input_part.detach().clone(),
                    "recurrent_part": recurrent_part.detach().clone(),
                    "hidden": hidden.detach().clone(),
                    "output": output_t.detach().clone(),
                })

        outputs = torch.stack(outputs, dim=1)
        hidden_states = torch.stack(hidden_states, dim=1)

        if return_debug:
            return outputs, hidden_states, hidden, debug_information

        return outputs, hidden_states, hidden



## 15. Inspect the model parameters

Because the matrices are wrapped in `nn.Parameter`, PyTorch knows they must be trained.

They are not separate visible layers.

They are trainable tensors stored inside the RNN module.


In [21]:

model = ManualRNN(
    input_size=vocabulary_size,
    hidden_size=3,
    output_size=vocabulary_size
)

for parameter_name, parameter in model.named_parameters():
    print(f"{parameter_name:5s} -> {tuple(parameter.shape)}")


W_xh  -> (5, 3)
W_hh  -> (3, 3)
b_h   -> (3,)
W_hy  -> (3, 5)
b_y   -> (5,)



## 16. Run a complete forward pass

The model returns:

1. `outputs`: logits at every time step
2. `hidden_states`: hidden state at every time step
3. `final_hidden`: the final hidden state

The final hidden state is useful for tasks where the whole sequence must be summarized, such as:

- sentiment classification
- sequence classification
- document classification

All time-step outputs are useful for tasks such as:

- next-word prediction
- sequence labeling
- part-of-speech tagging
- named-entity recognition


In [ ]:

outputs, hidden_states, final_hidden = model(x)

print("Input shape:", x.shape)
print("Outputs shape:", outputs.shape)
print("Hidden states shape:", hidden_states.shape)
print("Final hidden shape:", final_hidden.shape)



## 17. Inspect every internal calculation

The debug mode lets us see exactly what happened at each time step.

This is useful for learning and troubleshooting tensor shapes.


In [ ]:

outputs, hidden_states, final_hidden, debug_info = model(
    x,
    return_debug=True
)

for step in debug_info:
    time_step = step["time_step"]
    word_index = sequence_indices[0, time_step].item()
    word = index_to_word[word_index]

    print(f"Time step {time_step + 1}: {word}")
    print("x_t:", step["x_t"])
    print("x_t @ W_xh:", step["input_part"])
    print("h_previous @ W_hh:", step["recurrent_part"])
    print("new hidden:", step["hidden"])
    print("output logits:", step["output"])
    print("=" * 70)



# Training the manual RNN

We will train the RNN for next-word prediction.

Input:

```text
I like deep learning
```

Target:

```text
like deep learning .
```

At every time step, the model tries to predict the next word.

The input sequence has four time steps:

$$
[I,\ like,\ deep,\ learning]
$$

The target sequence also has four time steps:

$$
[like,\ deep,\ learning,\ .]
$$


In [ ]:

input_indices = torch.tensor([
    [
        vocabulary["I"],
        vocabulary["like"],
        vocabulary["deep"],
        vocabulary["learning"],
    ]
])

target_indices = torch.tensor([
    [
        vocabulary["like"],
        vocabulary["deep"],
        vocabulary["learning"],
        vocabulary["."],
    ]
])

training_x = F.one_hot(
    input_indices,
    num_classes=vocabulary_size
).float()

print("Training input shape:", training_x.shape)
print("Target shape:", target_indices.shape)



## 18. Why use `CrossEntropyLoss`?

At each time step, the model produces five logits:

$$
[\ell_1,\ell_2,\ell_3,\ell_4,\ell_5]
$$

Each logit corresponds to one possible next word.

`CrossEntropyLoss` compares these logits with the correct word index.

It internally applies the appropriate softmax-related calculation, so we should pass raw logits directly into it.

Do not manually apply softmax before `CrossEntropyLoss`.



## 19. Why reshape the outputs?

The model returns:

$$
(\text{batch size}, \text{sequence length}, \text{vocabulary size})
$$

For this example:

$$
(1,4,5)
$$

`CrossEntropyLoss` expects predictions shaped like:

$$
(\text{number of examples}, \text{number of classes})
$$

We therefore reshape:

$$
(1,4,5) \rightarrow (4,5)
$$

The targets are reshaped:

$$
(1,4) \rightarrow (4)
$$

Now each time step is treated as one classification example.


In [ ]:

training_model = ManualRNN(
    input_size=vocabulary_size,
    hidden_size=8,
    output_size=vocabulary_size
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    training_model.parameters(),
    lr=0.05
)

number_of_epochs = 500
loss_history = []

for epoch in range(number_of_epochs):
    optimizer.zero_grad()

    logits, hidden_states, final_hidden = training_model(training_x)

    loss = criterion(
        logits.reshape(-1, vocabulary_size),
        target_indices.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.6f}")



## 20. What happens during backpropagation?

The loss depends on the output logits.

The logits depend on:

$$
W_{hy}
$$

The hidden states depend on:

$$
W_{xh}, W_{hh}, b_h
$$

Because the hidden state at one time step influences later hidden states, gradients flow backward through the unrolled sequence.

This is called:

**Backpropagation Through Time**, or **BPTT**.

PyTorch automatically follows the computation graph across all time steps and calculates gradients for:

- $W_{xh}$
- $W_{hh}$
- $b_h$
- $W_{hy}$
- $b_y$

The optimizer then updates those parameters.


In [ ]:

print("Gradient shapes after training step:")

for parameter_name, parameter in training_model.named_parameters():
    gradient_shape = (
        None if parameter.grad is None
        else tuple(parameter.grad.shape)
    )
    print(f"{parameter_name:5s} -> {gradient_shape}")



## 21. Test the trained model

We choose the vocabulary word with the largest logit at each time step.

This is done using:

```python
argmax(dim=-1)
```


In [ ]:

training_model.eval()

with torch.no_grad():
    logits, _, _ = training_model(training_x)
    predicted_indices = logits.argmax(dim=-1)

input_words = [
    index_to_word[index]
    for index in input_indices[0].tolist()
]

target_words = [
    index_to_word[index]
    for index in target_indices[0].tolist()
]

predicted_words = [
    index_to_word[index]
    for index in predicted_indices[0].tolist()
]

print("Input words:     ", input_words)
print("Target words:    ", target_words)
print("Predicted words: ", predicted_words)



## 22. Inspect the learned probabilities

After training, each time step should assign a high probability to the correct next word.


In [ ]:

with torch.no_grad():
    logits, _, _ = training_model(training_x)
    probabilities = torch.softmax(logits, dim=-1)

for time_step in range(probabilities.shape[1]):
    current_word = input_words[time_step]
    target_word = target_words[time_step]

    step_probabilities = probabilities[0, time_step]

    print(f"After seeing: {current_word}")
    print(f"Correct next word: {target_word}")

    for word_index, probability in enumerate(step_probabilities):
        print(
            f"  {index_to_word[word_index]:8s}: "
            f"{probability.item():.4f}"
        )

    print("-" * 40)



# Important conceptual summary

## What changes at every time step?

The following values change:

$$
x_t
$$

the current input,

$$
h_t
$$

the current hidden state, and

$$
y_t
$$

the current output.

## What remains shared?

The same trainable parameters are reused:

$$
W_{xh}, W_{hh}, b_h, W_{hy}, b_y
$$

## Is $W_{hh}$ a separate layer?

No.

It is a trainable weight matrix **inside the recurrent layer**.

It is responsible for transforming the previous hidden state before that memory contributes to the current hidden state.

## Why is the hidden state useful?

It lets information from earlier time steps affect later predictions.

Without the recurrent term:

$$
h_{t-1}W_{hh}
$$

every word would be processed independently, and the network would no longer behave like a recurrent neural network.

## Why is the sequence length not part of the weight shape?

The same weights are reused at every time step.

A sequence of length 5 and a sequence of length 100 can use the same RNN parameters.

Sequence length controls how many times the recurrence is applied, not the dimensions of the trainable matrices.



# Shape reference

For batch size $B$, sequence length $T$, input size $I$, hidden size $H$, and output size $O$:

$$
x: B \times T \times I
$$

At one time step:

$$
x_t: B \times I
$$

$$
W_{xh}: I \times H
$$

$$
x_tW_{xh}: B \times H
$$

$$
h_{t-1}: B \times H
$$

$$
W_{hh}: H \times H
$$

$$
h_{t-1}W_{hh}: B \times H
$$

$$
h_t: B \times H
$$

$$
W_{hy}: H \times O
$$

$$
y_t: B \times O
$$

For all time steps:

$$
\text{hidden states}: B \times T \times H
$$

$$
\text{outputs}: B \times T \times O
$$



# Exercises

Try modifying the notebook to answer these questions:

1. What happens when `hidden_size` changes from `3` to `10`?
2. What shapes do $W_{xh}$, $W_{hh}$, and $W_{hy}$ get?
3. What happens if the initial hidden state is random instead of zero?
4. What happens if `tanh` is removed?
5. Can the model learn when the vocabulary contains more words?
6. How does the loss change when the learning rate is too high?
7. What happens if the sequence is processed in reverse order?
8. Can you use only the final hidden state for sentiment classification?
